# Held-out confirmation of generic+R, plan4_hold, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

`plan4_hold` only. Do **not** load `plan4_dev` or `plan4_fresh`. Do **not** overwrite `results_stage_plan4_hold`. Do not regenerate A. Do not train an adapter. Do not add calculations. Do not search more support variants.

These families were already evaluated in `results_stage_plan4_hold` on 13 Oct, with identical sender artifacts. This is the frozen candidate on previously evaluated holdout families, disjoint from development. It is not a fresh independent confirmation on untouched examples.

The candidate is frozen: independently prepared generic task-context instruction plus this example's transferred R. Construction, B prompts, scoring, and selection stay fixed.

1. **generic_r** — frozen candidate.
2. **result_kv** — reconstructed prefix plus transferred result states only.
3. **full_kv** — rebuilt full sender cache.
4. **text_result** — the corresponding RESULT lines as text.
5. **generic_mismatch** — generic support plus paired A-flip R; B's prompt stays this example's.

**Cell 4** (`--phase check`) rebuilds hold A from saved token IDs, prefills three generic templates, and checks R / mismatch / result-only prefix. Per-sheet `ok` must be a Boolean. If `all_passed` is false, stop.

**Cell 5** is 450 receiver evaluations.

Primary measure: all four eligibility labels. Also report whether mismatch predictions move toward the substituted labels.

Download `/kaggle/working/results_stage_plan4_confirm`. Leave `/kaggle/working/results_stage_plan4_hold` untouched.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_confirm.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_confirm.py").is_file() and (cand / "data" / "plan4_hold.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_confirm.py and data/plan4_hold.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_confirm.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
art = REPO / "results_stage_plan4_hold" / "a_artifacts.jsonl"
print("frozen hold A artifacts", art.is_file(), art)
hold = WORK / "results_stage_plan4_hold"
print("do not overwrite", hold)

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_confirm")
print("Run check first. Only compare if all_passed is true.")
print("Do not load plan4_dev. Do not overwrite results_stage_plan4_hold.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_confirm", "--split", "plan4_hold", "--phase", "check"])
print("plan4 confirm check exit", rc)
path = WORK / "results_stage_plan4_confirm" / "check_summary.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_confirm" / "run" / "check_summary.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    if not data.get("all_passed"):
        print("Rechecking construction ok from saved flags; do not rerun the model.")
        rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_confirm", "--split", "plan4_hold", "--phase", "rewrite_check"])
        print("plan4 confirm rewrite_check exit", rc)
        data = json.loads(path.read_text(encoding="utf-8"))
    rows = data.get("rows") or []
    if rows and not all(isinstance(row.get("ok"), bool) for row in rows):
        raise RuntimeError("Per-sheet ok must be a Boolean, not a partner id.")
    cons = data.get("construction") or {}
    keep = {
        "n": data.get("n"),
        "all_passed": data.get("all_passed"),
        "lead": data.get("lead"),
        "n_mismatch_ok": cons.get("n_mismatch_ok"),
        "n_result_kv_prefix_held": cons.get("n_result_kv_prefix_held"),
        "n_result_equal": cons.get("n_result_equal"),
        "templates": cons.get("templates"),
        "ok_types": sorted({type(row.get("ok")).__name__ for row in rows}),
        "n_span_extract_ok": sum(bool(row.get("span_extract_ok", True)) for row in rows),
    }
    print(json.dumps(keep, indent=2)[:16000])

In [ ]:
src = WORK / "results_stage_plan4_confirm" / "check_summary.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_confirm" / "run" / "check_summary.json"
check = json.loads(src.read_text(encoding="utf-8"))
if not check.get("all_passed"):
    raise RuntimeError("Check failed. Do not run compare.")
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_confirm", "--split", "plan4_hold", "--phase", "compare"])
print("plan4 confirm compare exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_confirm"
if not (src / "summary.json").is_file() and not (src / "check_summary.json").is_file():
    src = REPO / "results" / "stage_plan4_confirm" / "run"
print("reading", src)
for name in ("check_summary.json", "summary.json"):
    path = src / name
    if path.is_file():
        data = json.loads(path.read_text(encoding="utf-8"))
        print("\n====", name, "====")
        keep = {k: data.get(k) for k in ("n", "n_rows", "all_passed", "construction", "receiver", "label_shift", "family_clustered_eligibility", "lead") if k in data or data.get(k) is not None}
        if "receiver" in keep and isinstance(keep["receiver"], dict):
            keep["receiver"] = {
                cond: {kk: (keep["receiver"].get(cond) or {}).get(kk) for kk in ("n", "parse_ok", "truncated", "plan_all_status_correct", "correct", "correct_with_wrong_eligibility", "mean_n_kept", "mean_message_bytes", "mean_process_time_s")}
                for cond in keep["receiver"]
            }
        if "label_shift" in keep and isinstance(keep["label_shift"], dict):
            keep["label_shift"] = {
                cond: {kk: (keep["label_shift"].get(cond) or {}).get(kk) for kk in ("n", "parse_ok", "mean_n_match_this", "mean_n_match_src", "n_all_match_this", "n_all_match_src", "n_match_src_better", "n_result_len_differ", "correct_with_wrong_eligibility")}
                for cond in keep["label_shift"]
            }
        print(json.dumps(keep, indent=2)[:24000])